# BGG Ontology: From Natural Language to SPARQL

Two-call Claude pipeline:
1. User question â†’ Claude generates SPARQL
2. Execute SPARQL against GraphDB
3. Claude interprets results â†’ natural-language answer
 
**Prerequisites:** GraphDB running locally with the BGG ontology loaded (see Setup section below).
Set the `ANTHROPIC_API_KEY` environment variable before starting Jupyter.


## Setup

Load data into GraphDB before running queries:
1. Create a repository named `bgg` (or change `REPO_NAME` below)
2. Import `bgg_kaggle2025.ttl` into the default graph (126k games, 1.3M triples)
3. Optionally import `fake_players.ttl` into the same default graph (200 fake players)


## Configuration

In [1]:
import os
import json
import textwrap
from pathlib import Path
import requests
import anthropic

# --- GraphDB ---
GRAPHDB_URL     = "http://localhost:7200"
REPO_NAME       = "bgg"                     # change to match your repository name
SPARQL_ENDPOINT = f"{GRAPHDB_URL}/repositories/{REPO_NAME}"

# --- Anthropic ---
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    # Fallback: read from .env file (useful on Windows where env vars need a new process)
    _env_path = Path(os.getcwd()) / ".env"
    if _env_path.exists():
        for _line in _env_path.read_text().splitlines():
            if _line.startswith("ANTHROPIC_API_KEY="):
                ANTHROPIC_API_KEY = _line.split("=", 1)[1].strip().strip('"').strip("'")
                break

assert ANTHROPIC_API_KEY, (
    "ANTHROPIC_API_KEY not found. Either:\n"
    "  1. Restart VS Code after setting it in Windows environment settings, or\n"
    "  2. Create a .env file in this directory: ANTHROPIC_API_KEY=sk-ant-..."
)

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL  = "claude-opus-4-7"

print(f"GraphDB endpoint: {SPARQL_ENDPOINT}")
print(f"Model: {MODEL}")

GraphDB endpoint: http://localhost:7200/repositories/bgg
Model: claude-opus-4-7


## System Prompts

In [2]:
SYSTEM_SPARQL = """\
You are a SPARQL expert for a BoardGameGeek (BGG) knowledge graph stored in GraphDB.
Given a natural-language question, generate a single SPARQL 1.1 SELECT query that answers it.

## Board Game Knowledge Graph — Ontology Schema

### Namespaces (include in every query)
    PREFIX bgg:   <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX svj:   <https://vejdemo.se/boardgames#>
    PREFIX fake:  <https://vejdemo.se/boardgames/fake#>
    PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos:  <http://www.w3.org/2004/02/skos/core#>
    PREFIX xsd:   <http://www.w3.org/2001/XMLSchema#>

### Classes
- `bgg:Game` — A board game, tabletop game, or card game listed on BoardGameGeek — also called a title, game, or tabletop title.
- `bgg:Creator` — A person who designed, created, invented, authored, developed, or came up with a board game — also called a game designer, co-designer, author, or maker.
- `bgg:Category` — A thematic grouping, genre, type, style, or classification of a board game as defined by BGG — e.g. "Fantasy", "Card Game", "War Game". (see controlled vocabulary below)
- `bgg:Mechanic` — A gameplay mechanic, mechanism, rule system, or game system describing how a game is played — e.g. "Worker Placement", "Deck Building". Also called a gameplay element or game rule. (see controlled vocabulary below)
- `bgg:Publisher` — A company, studio, or individual that published, produced, released, or manufactured a board game.
- `bgg:Size` — The physical box size or weight class of a game: small (light, travel-sized), medium, or large (heavy, complex box).
- `bgg:MentalLoad` — The cognitive difficulty, complexity, or how hard a game is to learn and play: easy, moderate, or difficult.
- `bgg:Player` — A person who plays, owns, or collects board games — a gamer, board game player, or collector.
- `bgg:PlayerOpinion` — One player's personal opinion, review, rating, or assessment of a specific game.
- `trenj:Theme` — A thematic grouping, genre, topic, type, or what the game is about, as classified by the threnjen board game database — e.g. "Fantasy", "Steampunk", "Pirates". (see controlled vocabulary below)

### Datatype Properties on bgg:Game
- `bgg:hasName` xsd:string — The title, name, or what the game is called.
- `bgg:hasID` xsd:int — The BGG numeric ID, identifier, or BGG number.
- `bgg:hasDescription` xsd:string — A prose description, summary, overview, blurb, or "about" text for the game.
- `bgg:hasYearPublished` xsd:int — The year the game was published, released, came out, or first appeared.
- `bgg:hasRating` xsd:double — The average community rating or score on a 1–10 scale.
- `bgg:hasGeekRating` xsd:double — The BGG Geek Rating or BGG score.
- `bgg:ratingFromTime` xsd:int — The year the rating data was collected (2022 or 2025).
- `bgg:hasMinPlayers` xsd:int — The minimum number of players needed or required to play.
- `bgg:hasMaxPlayers` xsd:int — The maximum number of players supported, allowed, or accommodated.
- `bgg:hasBestNumPlayers` xsd:int — The optimal, recommended, or best player count.
- `bgg:hasMinGameTime` xsd:int — The minimum playtime, shortest duration, or quickest play time in minutes.
- `bgg:hasMaxGameTime` xsd:int — The maximum playtime, longest duration, or upper bound on play time in minutes.
- `bgg:hasMinRecAge` xsd:int — The minimum recommended age, age rating, or youngest suitable age in years.
- `bgg:isFullyEnriched` xsd:boolean — Whether the game has complete structured data: mechanics, categories, themes, creators, and publishers.

### Object Properties on bgg:Game
- `bgg:hasURL` URI — The BGG page link, web address, or URL for the game on BoardGameGeek.
- `bgg:hasThumbnail` URI — A thumbnail image, picture, cover art, or photo URL.
- `bgg:hasCategory` bgg:Category — The category, genre, type, or classification of the game. (match via rdfs:label)
- `bgg:hasMechanic` bgg:Mechanic — The gameplay mechanic, mechanism, or game system. (match via rdfs:label)
- `bgg:hasCreator` bgg:Creator — The designer, creator, or author of the game. (match via rdfs:label)
- `bgg:hasPublisher` bgg:Publisher — The publisher, company, or studio that produced the game. (match via rdfs:label)
- `bgg:hasSize` bgg:Size — box size: bgg:small / bgg:medium / bgg:large
- `trenj:hasTheme` trenj:Theme — The theme, topic, setting, or what the game is about. (match via rdfs:label)

### Object Properties on bgg:Player / bgg:PlayerOpinion
- `bgg:hasOwnershipOf` — A game that the player owns, has, or keeps in their collection.
- `bgg:hasOpinionHolder` — The player who holds, gave, or wrote this opinion or review.
- `bgg:hasOpinionOf` — The game this opinion, review, or rating is about.
- `bgg:hasMentalLoad` — The cognitive difficulty or complexity this player assigned: bgg:easy / bgg:moderate / bgg:difficult.
- `bgg:hasPlayerRatingOpinion` xsd:decimal — A player's personal rating, score, or how much they liked a game on a 1–10 scale.
- `bgg:likesCategory` — A category, genre, or type the player likes, enjoys, or prefers.
- `bgg:likesMechanic` — A mechanic or game system the player likes, enjoys, or prefers.

### Controlled Vocabulary — exact rdfs:label values

Use the vocabulary lists to identify the right concept name, then filter with CONTAINS(LCASE(str(?label)), LCASE("term")).
This handles variant and overlapping labels — e.g. CONTAINS("action point") matches both "Action Points" and "Action Point Allowance System".
Vocabulary instances also carry skos:altLabel for common synonyms and abbreviations (e.g. "WW1", "sci-fi", "co-op").
Use the rdfs:label|skos:altLabel property path to match both: ?v rdfs:label|skos:altLabel ?label .

**bgg:Category labels (89):**
"Abstract Strategy", "Action Dexterity", "Adventure", "Age of Reason", "American Civil War", "American Indian Wars", "American Revolutionary War", "American West", "Ancient", "Animals", "Arabian", "Aviation / Flight", "Aviation Flight", "Bluffing", "Book", "Card Game", "Childrens Game", "City Building", "Civil War", "Civilization", "Collectible Components", "Comic Book / Strip", "Comic Book Strip", "Deduction", "Dice", "Economic", "Educational", "Electronic", "Environmental", "Expansion for Base-game", "Exploration", "Fantasy", "Farming", "Fighting", "Game System", "Horror", "Humor", "Industry / Manufacturing", "Industry Manufacturing", "Korean War", "Mafia", "Math", "Mature / Adult", "Mature Adult", "Maze", "Medical", "Medieval", "Memory", "Miniatures", "Modern Warfare", "Movies / TV / Radio theme", "Movies TV Radio theme", "Murder Mystery", "Music", "Mythology", "Napoleonic", "Nautical", "Negotiation", "Novel-based", "Number", "Party Game", "Pike and Shot", "Pirates", "Political", "Post-Napoleonic", "Prehistoric", "Print & Play", "Puzzle", "Racing", "Real-Time", "Real-time", "Religious", "Renaissance", "Science Fiction", "Space Exploration", "Spies / Secret Agents", "Sports", "Territory Building", "Trains", "Transportation", "Travel", "Trivia", "Video Game Theme", "Vietnam War", "Wargame", "Word Game", "World War I", "World War II", "Zombies"

**bgg:Mechanic labels (176):**
"Acting", "Action Drafting", "Action Movement Programming", "Action Point Allowance System", "Action Points", "Action Queue", "Action Retrieval", "Action Timer", "Action/Event", "Advantage Token", "Alliances", "Area Control Area Influence", "Area Enclosure", "Area Majority / Influence", "Area Movement", "Area-Impulse", "Auction/Bidding", "Auction / Bidding", "Automatic Resource Growth", "Betting and Bluffing", "Betting / Wagering", "Bias", "Bingo", "Bribery", "Campaign / Battle Card Driven", "Campaign Battle Card Driven", "Card Drafting", "Card Play Conflict Resolution", "Catch the Leader", "Chaining", "Chit-Pull System", "Co-operative Play", "Command Cards", "Commodity Speculation", "Communication Limits", "Connections", "Contracts", "Cooperative Game", "Crayon Rail System", "Critical Hits and Failures", "Cube Tower", "Deck Construction", "Deck Pool Building", "Deck, Bag, and Pool Building", "Deduction", "Delayed Purchase", "Dexterity", "Dice Rolling", "Die Icon Resolution", "Different Dice Movement", "Drafting", "Elapsed Real Time Ending", "Enclosure", "End Game Bonuses", "Events", "Finale Ending", "Follow", "Force Commitment", "Grid Coverage", "Grid Movement", "Hand Management", "Hex-and-Counter", "Hexagon Grid", "Hidden Movement", "Hidden Roles", "Hidden Victory Points", "Highest-Lowest Scoring", "I Cut, You Choose", "Impulse Movement", "Income", "Increase Value of Unchosen Resources", "Induction", "Interrupts", "Investment", "Kill Steal", "King of the Hill", "Ladder Climbing", "Layering", "Legacy Game", "Line Drawing", "Line of Sight", "Loans", "Lose a Turn", "Mancala", "Map Addition", "Map Deformation", "Map Reduction", "Market", "Matching", "Measurement Movement", "Melding and Splaying", "Memory", "Minimap Resolution", "Modular Board", "Move Through Deck", "Movement Points", "Movement Template", "Moving Multiple Units", "Multiple Maps", "Narrative Choice / Paragraph", "Negotiation", "Network and Route Building", "Once-Per-Game Abilities", "Order Counters", "Ownership", "Paper-and-Pencil", "Partnerships", "Passed Action Token", "Pattern Building", "Pattern Movement", "Pattern Recognition", "Physical", "Physical Removal", "Pick-up and Deliver", "Pieces as Map", "Player Elimination", "Player Judge", "Point to Point Movement", "Predictive Bid", "Press Your Luck", "Prisoner's Dilemma", "Programmed Movement", "Push Your Luck", "Race", "Random Production", "Ratio / Combat Results Table", "Re-rolling and Locking", "Real-Time", "Real-time", "Relative Movement", "Resource to Move", "Rock-Paper-Scissors", "Role Playing", "Roles with Asymmetric Information", "Roll / Spin and Move", "Roll Spin and Move", "Rondel", "RouteNetwork Building", "Scenario / Mission / Campaign Game", "Score-and-Reset Game", "Secret Unit Deployment", "Semi-Cooperative Game", "Set Collection", "Simulation", "Simultaneous Action Selection", "Singing", "Single Loser Game", "Slide/Push", "Solo / Solitaire Game", "Speed Matching", "Square Grid", "Stat Check Resolution", "Static Capture", "Stock Holding", "Storytelling", "Sudden Death Ending", "Tableau Building", "Take That", "Targeted Clues", "Team-Based Game", "Tech Trees / Tech Tracks", "Three Dimensional Movement", "Tile Placement", "Time Track", "Track Movement", "Trading", "Traitor Game", "Trick-taking", "Tug of War", "Variable Phase Order", "Variable Player Powers", "Variable Set-up", "Victory Points as a Resource", "Voting", "Worker Placement", "Zone of Control"

**trenj:Theme labels (222):**
"Adventure", "Age of Reason", "American Civil War", "American Indian Wars", "American Revolutionary War", "American West", "Ancient", "Animals", "Arabian", "Aviation / Flight", "Aviation Flight", "City Building", "Civil War", "Civilization", "Comic Book / Strip", "Comic Book Strip", "Economic", "Environmental", "Fantasy", "Farming", "Fighting", "Horror", "Humor", "Industry / Manufacturing", "Industry Manufacturing", "Korean War", "Mafia", "Math", "Mature / Adult", "Mature Adult", "Maze", "Medical", "Medieval", "Modern Warfare", "Movies / TV / Radio theme", "Movies TV Radio theme", "Murder Mystery", "Music", "Mythology", "Napoleonic", "Nautical", "Novel-based", "Number", "Pike and Shot", "Pirates", "Political", "Post-Napoleonic", "Prehistoric", "Racing", "Religious", "Renaissance", "Science Fiction", "Space Exploration", "Spies / Secret Agents", "Sports", "African Americans", "Airships / Blimps / Dirigibles / Zeppelins", "Alchemy", "Alternate History", "Amusement Parks / Theme Parks", "Animal Battles", "Anime / Manga", "Anthropomorphic Animals", "Apache Tribes", "Arcade Video Games", "Archaeology / Paleontology", "Art", "Astrology", "Astronomy", "Attorneys / Courts", "Automotive Industry", "Aztecs", "Bacteria", "Battle Royal", "Beaches", "Biology", "Boardgaming", "Books / Libraries", "Camping", "Canals", "Cannibals / Cannibalis", "Care Bears", "Cemeteries / Graveyards", "Cereal Games", "Chemistry", "Chernobyl", "Chibis", "Chivalry / Jousting / Tournaments (Medieval Europe)", "Circus", "City", "Climate Chang", "Colonial", "Computer / Information Technology Industry", "Construction", "Cruise ships", "Cthulhu Mythos", "Cyberpunk", "Deserts", "Dieselpunk", "Disney Theme Parks", "Dolls", "Dreams / Nightmares", "Druids", "Earthquakes", "Ecology", "Evolution", "FIFA World Cup", "Fantasy Sports", "Fashion", "Fictional Games", "Firefighting", "Flags identification", "Floating islands in the sky", "Flowers", "Food / Cooking", "French Foreign Legion", "Fruit", "Gardening", "Geocaching", "Gladiators", "Hackers", "Hanseatic Leagu", "Helicopters", "Hell", "Hike-Hiking", "Hot Air Balloons", "Inuit Peoples", "Jail / Prison (Modern)", "Jewelry", "Journalis", "Kaiju", "King Arthur / The Knights of the Round Table / Camelot", "Knights Templar", "Latin American Political Games", "Love / Romanc", "Mad Science / Mad Scientist", "Mail / Stamps / The Post Offic", "Mayans", "Mech Warfar", "Memes", "Mining", "Motorcycles", "Movie Industry", "Movies", "Mushrooms", "Music Making & Makers", "My Best Lif", "Mystery / Cri", "Māori", "Native Americans / First Peoples", "Natur", "Ninjas", "Nuclear option", "Oil / Gas / Petroleu", "Painting / Paintings", "Perfu", "Photography", "Polic", "Post-Apocalyptic", "Psychic Powers", "Psychology", "Pub, Bars, Bistros...", "Pulp", "Retro", "Rivers", "Robots", "Romance of the Three Kingdoms", "Rubik's Cub", "Safaris", "Samurai", "School / College / University", "Sci-Fi Sports", "Scienc", "Sewing / Knitting / Cloth-Making", "Sieg", "Silk Road", "Spanish Political Games", "Spooky Old Houses", "Steampunk", "Submarines", "Superheroes", "Survival", "TV Detectives", "Teaching Programming", "Television (TV) Industry", "Templ", "Tiki Cultur", "Time Travel", "Traffic / Driving", "Trash / Garbag", "Trees and Forests", "Tropical", "Tropical Islands", "Trucks", "UFOs", "US National Parks", "Under the Sea", "Video Game Theme: Carmen Sandiego", "Video Game Theme: Doo", "Video Game Theme: Final Fantasy", "Video Game Theme: Minecraft", "Video Game Theme: Nintendo", "Video Game Theme: Pokémon", "Video Game Theme: Resident Evil", "Video Game Theme: SEGA", "Video Game Theme: Super Mario Bros.", "Video Game Theme: Tetris", "Video Game Theme: The Oregon Trail", "Vikings", "Villainy", "Volcanoes", "Weather", "Witches", "Trains", "Transportation", "Travel", "Trivia", "Video Game Theme", "Vietnam War", "World War I", "World War II", "Zombies"

### IMPORTANT DATA NOTES
1. Only 21,379 games have `bgg:isFullyEnriched true` and carry mechanics/categories/themes/creators/publishers.
   The other ~105k games have only: name, description, year, ratings, thumbnail, URL.
2. Always use an exact label from the controlled vocabulary lists above when filtering by category, mechanic, or theme.
3. `bgg:hasName` values are xsd:string — match with "Catan"^^xsd:string or FILTER(str(?name) = "Catan").
4. For case-insensitive name search: FILTER(CONTAINS(LCASE(str(?name)), "catan")).
5. Fake player data (200 players, namespace fake:) and Susanne's data (namespace svj:) are in the same graph.

### Example SPARQL Queries

Find a game by exact name:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?game ?year ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName "Catan"^^xsd:string ;
            bgg:hasYearPublished ?year ;
            bgg:hasRating ?rating .
    }

Top-rated games for 2 players:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating ;
            bgg:hasMinPlayers ?min ;
            bgg:hasMaxPlayers ?max .
      FILTER(?min <= 2 && ?max >= 2)
    }
    ORDER BY DESC(?rating) LIMIT 10

Games with Worker Placement mechanic (CONTAINS for label matching):
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasMechanic ?mech ;
            bgg:hasRating ?rating .
      ?mech rdfs:label ?mechLabel .
      FILTER(CONTAINS(LCASE(str(?mechLabel)), LCASE("worker placement")))
    }
    ORDER BY DESC(?rating) LIMIT 10

Fake player opinions:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?playerName ?gameName ?personalRating WHERE {
      ?op a bgg:PlayerOpinion ;
          bgg:hasOpinionHolder ?player ;
          bgg:hasOpinionOf ?game ;
          bgg:hasPlayerRatingOpinion ?personalRating .
      ?player rdfs:label ?playerName .
      ?game bgg:hasName ?gameName .
      FILTER(?personalRating = 10)
    } LIMIT 20

Games about a topic — always check both Category and Theme using UNION with CONTAINS:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX trenj: <https://vejdemo.se/boardgames/threnjen#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
      { ?game bgg:hasCategory ?tag . ?tag rdfs:label ?tagLabel .
        FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
      UNION
      { ?game trenj:hasTheme ?tag . ?tag rdfs:label ?tagLabel .
        FILTER(CONTAINS(LCASE(str(?tagLabel)), LCASE("fantasy"))) }
    }
    ORDER BY DESC(?rating) LIMIT 10

Games matching a concept across both categories and mechanics — UNION with altLabel property path:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating .
      { ?game bgg:hasCategory ?v } UNION { ?game bgg:hasMechanic ?v }
      ?v rdfs:label|skos:altLabel ?label .
      FILTER(CONTAINS(LCASE(str(?label)), LCASE("deduction")))
    }
    ORDER BY DESC(?rating) LIMIT 10

Game name and creator if known — OPTIONAL for enriched-only properties:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?name ?year ?rating ?creatorName WHERE {
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasYearPublished ?year ;
            bgg:hasRating ?rating .
      OPTIONAL {
        ?game bgg:hasCreator ?creator .
        ?creator rdfs:label ?creatorName .
      }
    }
    ORDER BY DESC(?rating) LIMIT 10

Most common mechanics — GROUP BY with HAVING:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?mechName (COUNT(?game) AS ?gameCount) (AVG(?rating) AS ?avgRating) WHERE {
      ?game a bgg:Game ;
            bgg:hasMechanic ?mech ;
            bgg:hasRating ?rating .
      ?mech rdfs:label ?mechName .
    }
    GROUP BY ?mechName
    HAVING (COUNT(?game) > 10)
    ORDER BY DESC(?gameCount) LIMIT 10

Player recommendations — games matching liked categories not yet owned:
    PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name ?rating WHERE {
      ?player a bgg:Player ;
              rdfs:label "Alice"@en ;
              bgg:likesCategory ?cat .
      ?game a bgg:Game ;
            bgg:hasName ?name ;
            bgg:hasRating ?rating ;
            bgg:hasCategory ?cat .
      FILTER NOT EXISTS { ?player bgg:hasOwnershipOf ?game }
    }
    ORDER BY DESC(?rating) LIMIT 10

Rules:
- Always include all required PREFIX declarations.
- Return ONLY the raw SPARQL query with no explanation, no markdown fences, no commentary.
- Use OPTIONAL { } for properties that may not exist on all games (mechanics, categories, themes, creators, publishers only exist for the 21,379 fully enriched games).
- Use ORDER BY DESC(?rating) and LIMIT when asked for 'best', 'top', or 'most popular'.
- When filtering by category, mechanic, or theme, use CONTAINS(LCASE(str(?label)), LCASE("term")) rather than exact match, to handle variant and overlapping labels. Use the vocabulary lists to choose an appropriate search term.
- When the user asks about topic, theme, genre, setting, or what a game is about, always query BOTH bgg:hasCategory and trenj:hasTheme using a UNION.
- When the user's term could describe either a category or a mechanic (e.g. "deduction game", "bluffing game", "cooperative game", "racing game"), also UNION across bgg:hasCategory and bgg:hasMechanic using the rdfs:label|skos:altLabel property path.
- When altLabels are present, use rdfs:label|skos:altLabel ?label in place of rdfs:label ?label to match both standard and alternative labels.
- For player count: "N players" / "for N players" / "playable by N" means N is within the valid range: FILTER(?min <= N && ?max >= N). "Exactly N players" / "strictly N" / "only N players" means the game is designed for that count only: FILTER(?min = N && ?max = N). "At least N players" means FILTER(?min <= N). "At most N players" / "up to N players" means FILTER(?max <= N).
- For free-text game name search: FILTER(CONTAINS(LCASE(str(?name)), LCASE("term"))).
- For exact game name: bgg:hasName "Title"^^xsd:string.
- Do not use DESCRIBE or CONSTRUCT — only SELECT queries.
- Avoid Cartesian products; use FILTER close to the relevant triple pattern.
"""

SYSTEM_RESULT = """\
You are a helpful assistant explaining SPARQL query results about board games.
Given a question and the raw SPARQL results (as JSON), write a clear, concise natural-language answer.
Focus on answering the question directly. If the results are empty, say so and suggest why.
If results_truncated is true, note that only the top 50 of result_count results are shown.
Format lists nicely. Round numbers to 2 decimal places. Keep your answer under 200 words.
"""

print(f"SPARQL system prompt: {len(SYSTEM_SPARQL)} chars")
print(f"Result system prompt: {len(SYSTEM_RESULT)} chars")

# Alternative label-matching method — exact match (swap in to compare against CONTAINS method above).
# To test: replace the vocabulary intro note, the two label-filtering examples, and the rule below.
PROMPT_ADDITION_ALTERNATIVE = """
# Vocabulary intro (replaces current CONTAINS note in SYSTEM_SPARQL):
When filtering by category, mechanic, or theme, use an exact label from these lists.
Map user terms to the closest matching label (e.g. "sci-fi" → "Science Fiction", "pirates" → "Pirates", "steampunk" → "Steampunk").

# Example — Worker Placement (exact match, replaces CONTAINS version):
      ?mech rdfs:label "Worker Placement"@en .

# Example — UNION topic query (exact match, replaces CONTAINS version):
      { ?game bgg:hasCategory ?tag . ?tag rdfs:label "Fantasy"@en }
      UNION
      { ?game trenj:hasTheme ?tag . ?tag rdfs:label "Fantasy"@en }

# Rule (replaces CONTAINS rule in Rules section):
- When filtering by category, mechanic, or theme, always use an exact label from the
  controlled vocabulary lists. Map user terms to the closest matching label.
"""

SPARQL system prompt: 22893 chars
Result system prompt: 455 chars


## Pipeline Functions

In [3]:
MAX_RETRIES = 2


def generate_sparql(question: str, error_context: str = None) -> str:
    """Call 1: natural language -> SPARQL query. Pass error_context to repair a bad query."""
    user_content = question
    if error_context:
        nl = chr(10)
        user_content = (nl * 2).join([
            question,
            f"Your previous query failed with this GraphDB error: {error_context}",
            "Please fix the SPARQL query.",
        ])
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=[
            {
                "type": "text",
                "text": SYSTEM_SPARQL,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[{"role": "user", "content": user_content}],
    )
    query = response.content[0].text.strip()
    # Strip markdown fences if present
    if query.startswith("```"):
        query = chr(10).join(
            l for l in query.splitlines() if not l.startswith("```")
        ).strip()
    return query


def execute_sparql(query: str) -> dict:
    """Execute a SPARQL SELECT against GraphDB; return parsed JSON results."""
    resp = requests.post(
        SPARQL_ENDPOINT,
        data={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def format_results_table(results: dict, max_rows: int = 20) -> str:
    """Format SPARQL JSON results as a markdown table (values truncated at 80 chars)."""
    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head",    {}).get("vars",     [])
    if not bindings or not vars_:
        return "(no results)"

    def cell(b, v):
        val = b.get(v, {}).get("value", "")
        return val[:80] + "..." if len(val) > 80 else val

    rows  = [[cell(b, v) for v in vars_] for b in bindings]
    shown = rows[:max_rows]
    col_w = [max(len(v), max(len(r[i]) for r in shown)) for i, v in enumerate(vars_)]

    def md_row(cells):
        return "| " + " | ".join(c.ljust(col_w[i]) for i, c in enumerate(cells)) + " |"

    header  = md_row(vars_)
    divider = "| " + " | ".join("-" * col_w[i] for i in range(len(vars_))) + " |"
    lines   = [header, divider] + [md_row(r) for r in shown]
    if len(bindings) > max_rows:
        lines.append(f"*(showing {max_rows} of {len(bindings)} rows)*")
    return chr(10).join(lines)


def interpret_results(question: str, query: str, results: dict) -> str:
    """Call 2: SPARQL results -> natural-language answer."""
    bindings = results.get("results", {}).get("bindings", [])
    vars_    = results.get("head",    {}).get("vars",     [])
    truncated = len(bindings) > 50

    payload = {
        "question": question,
        "sparql_query": query,
        "result_variables": vars_,
        "result_count": len(bindings),
        "results_truncated": truncated,
        "results": bindings[:50],
    }

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_RESULT,
        messages=[
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            }
        ],
    )
    return response.content[0].text.strip()


def ask(question: str, show_query: bool = True) -> str:
    """Full pipeline: question -> SPARQL -> execute (with retry) -> natural-language answer."""
    print(f"Question: {question}")
    print()

    query = generate_sparql(question)
    for attempt in range(MAX_RETRIES + 1):
        if show_query:
            label = f"Generated SPARQL (attempt {attempt + 1}):" if attempt > 0 else "Generated SPARQL:"
            print(label)
            print("-" * 60)
            print(query)
            print("-" * 60)
            print()

        try:
            results = execute_sparql(query)
            break
        except requests.HTTPError as e:
            error_msg = e.response.text[:500]
            if attempt < MAX_RETRIES:
                print(f"SPARQL error (attempt {attempt + 1}), retrying...")
                print(error_msg[:120])
                print()
                query = generate_sparql(question, error_msg)
            else:
                print(f"SPARQL execution failed after {MAX_RETRIES + 1} attempts.")
                print(error_msg)
                return ""

    n = len(results.get("results", {}).get("bindings", []))
    print(f"Results: {n} row(s)" + (" (truncated to 50 for interpretation)" if n > 50 else ""))
    print()
    print(format_results_table(results))
    print()

    answer = interpret_results(question, query, results)
    print("Answer:")
    print(textwrap.fill(answer, width=80))
    print()
    return answer

## Test GraphDB Connectivity

In [4]:
test_query = "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
try:
    result = execute_sparql(test_query)
    count = result["results"]["bindings"][0]["n"]["value"]
    print(f"GraphDB is reachable. Total triples: {count}")
except Exception as e:
    print(f"Cannot reach GraphDB: {e}")
    print(f"Make sure GraphDB is running at {GRAPHDB_URL}")
    print(f"and a repository named '{REPO_NAME}' exists with the TTL loaded.")


GraphDB is reachable. Total triples: 579510


## Example Queries

Run any cell below, or write your own at the bottom.


In [5]:
ask("What are the top 10 highest-rated board games of all time?")

Question: What are the top 10 highest-rated board games of all time?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?rating ?year WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasYearPublished ?year .
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                                  | rating | year |
| ----------------------------------------------------- | ------ | ---- |
| TerroriXico                                           | 9.43   | 2018 |
| Meeple Quest                                          | 9.32   | 2021 |
| The Fantasy Trip: Legacy Edition                      | 9.29   | 2019 |
| Wings of the Motherland                               | 9.17   | 2019 |
| 1985: Deadly Northern Lights                          | 9.17   | 202

"Here are the top 10 highest-rated board games of all time:\n\n1. **TerroriXico** (2018) — 9.43\n2. **Meeple Quest** (2021) — 9.32\n3. **The Fantasy Trip: Legacy Edition** (2019) — 9.29\n4. **Wings of the Motherland** (2019) — 9.17\n5. **1985: Deadly Northern Lights** (2020) — 9.17\n6. **System Gateway (fan expansion for Android: Netrunner)** (2021) — 9.16\n7. **Global War 1939** (2011) — 9.09\n8. **Animator vs Animation** (2018) — 9.02\n9. **World in Flames Collector's Edition Deluxe** (2017) — 9.01\n10. **Urgency** (2020) — 9.00\n\nAll of these games boast exceptional ratings above 9.0, with most being relatively recent releases from 2017–2021. Note that these ratings may reflect smaller, enthusiastic communities rather than mainstream popularity."

In [6]:
ask("Which games use the Worker Placement mechanic and have a rating above 8?")

Question: Which games use the Worker Placement mechanic and have a rating above 8?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMechanic ?mech .
  ?mech rdfs:label|skos:altLabel ?mechLabel .
  FILTER(CONTAINS(LCASE(str(?mechLabel)), LCASE("worker placement")))
  FILTER(?rating > 8)
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 35 row(s)

| name                                                                       | rating |
| -------------------------------------------------------------------------- | ------ |
| Anachrony: Infinity Box                                                    | 8.97   |
| Tr

"There are **35 games** using the Worker Placement mechanic with a rating above 8. Here are the top-rated ones:\n\n1. **Anachrony: Infinity Box** – 8.97\n2. **Trickerion: Collector's Edition** – 8.76\n3. **Everdell: Collector's Edition** – 8.52\n4. **The Castles of Burgundy** – 8.44\n5. **Dune: Imperium** – 8.43\n6. **Kanban EV** – 8.41\n7. **Lorenzo il Magnifico: Big Box** – 8.40\n8. **Ultimate Railroads** – 8.39\n9. **Viticulture: Complete Collector's Edition** – 8.32\n10. **On Mars** – 8.18\n\nOther notable entries include **A Feast for Odin** (8.17), **Barrage** (8.16), **Dwellings of Eldervale** (8.13), **Lost Ruins of Arnak** (8.07), **Orléans** (8.05), **Underwater Cities** (8.05), **The Gallerist** (8.02), and **Everdell** (8.01).\n\nWorker placement clearly remains one of the most beloved mechanics in heavy strategy gaming, with strong representation from designers like Vital Lacerda and Uwe Rosenberg."

In [7]:
ask("What games can be played by exactly 2 players and take less than 30 minutes?")

Question: What games can be played by exactly 2 players and take less than 30 minutes?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
SELECT ?name ?rating ?minTime ?maxTime WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max ;
        bgg:hasMinGameTime ?minTime ;
        bgg:hasMaxGameTime ?maxTime .
  FILTER(?min = 2 && ?max = 2)
  FILTER(?maxTime < 30)
}
ORDER BY DESC(?rating) LIMIT 20
------------------------------------------------------------

Results: 20 row(s)

| name                                                                                | rating | minTime | maxTime |
| ----------------------------------------------------------------------------------- | ------ | ------- | ------- |
| Revanche                                                                            | 8.

'Here are the top-rated games designed strictly for 2 players with a maximum playtime under 30 minutes:\n\n1. **Revanche** — 8.46 (10–20 min)\n2. **BattleCON: Wanderers of Indines** — 8.36 (25 min)\n3. **Exceed: Street Fighter – Chun-Li Box** — 8.24 (15–25 min)\n4. **Exceed: Shovel Knight – Shadow Box** — 8.23 (15–25 min)\n5. **Exceed: Street Fighter – M. Bison Box** — 8.22 (15–25 min)\n6. **Unmatched: Little Red Riding Hood vs. Beowulf** — 8.11 (20 min)\n7. **Exceed: Shovel Knight – Hope Box** — 8.10 (15–25 min)\n8. **Unmatched: Robin Hood vs. Bigfoot** — 7.92 (20 min)\n9. **Onager** — 7.88 (20 min)\n10. **Exceed: Red Horizon – Satoshi, Mei Lien, Morathi, and Baelkhor** — 7.83 (10–20 min)\n\nOther notable entries include **Star Realms: Colony Wars**, **Hive Pocket**, **Cuttle**, and **Final Fantasy VIII: Triple Triad**. The list is dominated by dueling card games — particularly the *Exceed* and *Unmatched* series.'

In [8]:
ask("What games did Uwe Rosenberg design?")

Question: What games did Uwe Rosenberg design?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?year ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasCreator ?creator .
  OPTIONAL { ?game bgg:hasYearPublished ?year . }
  OPTIONAL { ?game bgg:hasRating ?rating . }
  ?creator rdfs:label ?creatorName .
  FILTER(CONTAINS(LCASE(str(?creatorName)), LCASE("Uwe Rosenberg")))
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 67 row(s) (truncated to 50 for interpretation)

| name                                                | year | rating |
| --------------------------------------------------- | ---- | ------ |
| A Feast for Odin                                    | 2016 | 8.17   |
| Fields of Arle                                      | 2014 | 8

"Uwe Rosenberg has designed 67 games in the database (showing the top 50 by rating). He's best known for heavy Euro-style games, with his highest-rated designs being:\n\n**Top 10 by rating:**\n1. **A Feast for Odin** (2016) — 8.17\n2. **Fields of Arle** (2014) — 8.03\n3. **Agricola (Revised Edition)** (2016) — 7.95\n4. **Caverna: The Cave Farmers** (2013) — 7.94\n5. **Agricola** (2007) — 7.87\n6. **Hallertau** (2020) — 7.86\n7. **Le Havre** (2008) — 7.84\n8. **Ora et Labora** (2011) — 7.68\n9. **Nusfjord** (2017) — 7.64\n10. **Agricola: All Creatures Big and Small – The Big Box** (2018) — 7.64\n\nOther notable designs include **Patchwork** (7.59), **Glass Road** (7.42), and the popular card game **Bohnanza** (7.07), along with numerous Bohnanza spin-offs. His catalog spans from early 1990s titles like *Times* (1992) to recent releases like *New York Zoo* and *Fairy Trails* (2020).\n\nNote: Only the top 50 of 67 results are shown."

In [ ]:
ask("What games did Uwe Rossenberg design?")
### Intentional misspelling of name 

Question: What games did Uwe Rossenberg design?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?year ?rating ?creatorName WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasCreator ?creator .
  ?creator rdfs:label ?creatorName .
  OPTIONAL { ?game bgg:hasYearPublished ?year . }
  OPTIONAL { ?game bgg:hasRating ?rating . }
  FILTER(CONTAINS(LCASE(str(?creatorName)), LCASE("Rosenberg")) || CONTAINS(LCASE(str(?creatorName)), LCASE("Rossenberg")))
}
ORDER BY DESC(?rating)
------------------------------------------------------------

Results: 67 row(s) (truncated to 50 for interpretation)

| name                                                | year | rating | creatorName   |
| --------------------------------------------------- | ---- | ------ | ------------- |
| A Feast for Odin                 

'Uwe Rosenberg (note: the correct spelling is "Rosenberg," not "Rossenberg") has designed **67 games** in the database. Showing the top 50 by rating, here are some highlights:\n\n**Top-rated designs:**\n- **A Feast for Odin** (2016) – 8.17\n- **Fields of Arle** (2014) – 8.03\n- **Agricola (Revised Edition)** (2016) – 7.95\n- **Caverna: The Cave Farmers** (2013) – 7.94\n- **Agricola** (2007) – 7.87\n- **Hallertau** (2020) – 7.86\n- **Le Havre** (2008) – 7.84\n- **Ora et Labora** (2011) – 7.68\n- **Nusfjord** (2017) – 7.64\n- **Patchwork** (2014) – 7.59\n\n**Other notable titles:** Glass Road, At the Gates of Loyang, Nova Luna, Bohnanza (and many Bohnanza variants/expansions), Cottage Garden, Indian Summer, Reykholt, Babel, Mamma Mia!, and Second Chance.\n\nRosenberg is best known for heavy Euro-style farming/economic games (the Agricola/Caverna/Le Havre family) as well as lighter card games like the long-running Bohnanza series. Note: results are truncated — 17 additional lower-rated ti

In [9]:
ask("Which categories have the most games?")

Question: Which categories have the most games?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?catName (COUNT(?game) AS ?gameCount) WHERE {
  ?game a bgg:Game ;
        bgg:hasCategory ?cat .
  ?cat rdfs:label ?catName .
}
GROUP BY ?catName
ORDER BY DESC(?gameCount)
LIMIT 20
------------------------------------------------------------

Results: 20 row(s)

| catName            | gameCount |
| ------------------ | --------- |
| Card Game          | 6300      |
| Card_Game          | 6300      |
| Miniatures         | 1048      |
| Wargame            | 945       |
| Exploration        | 859       |
| Fantasy            | 717       |
| Puzzle             | 651       |
| Print & Play       | 575       |
| Print_&_Play       | 575       |
| Economic           | 554       |
| Fighting           | 518       |
| Territory Building | 49

'The categories with the most games are:\n\n1. **Card Game** – 6,300 games\n2. **Miniatures** – 1,048 games\n3. **Wargame** – 945 games\n4. **Exploration** – 859 games\n5. **Fantasy** – 717 games\n6. **Puzzle** – 651 games\n7. **Print & Play** – 575 games\n8. **Economic** – 554 games\n9. **Fighting** – 518 games\n10. **Territory Building** – 496 games\n11. **Educational** – 476 games\n12. **Word Game** – 473 games\n13. **Science Fiction** – 464 games\n14. **Dice** – 353 games\n15. **World War II** – 340 games\n\nCard Game dominates by a wide margin, with over six times as many games as the next category. Note: some categories appear duplicated in the raw results due to label variants (e.g., "Card Game" vs "Card_Game"); I\'ve consolidated those above.'

In [16]:
ask("What games are in the Fantasy category with a geek rating above 7.5?")

Question: What games are in the Fantasy category with a geek rating above 7.5?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?geekRating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasGeekRating ?geekRating ;
        bgg:hasCategory ?cat .
  ?cat rdfs:label ?catLabel .
  FILTER(CONTAINS(LCASE(str(?catLabel)), LCASE("fantasy")))
  FILTER(?geekRating > 7.5)
}
ORDER BY DESC(?geekRating)
------------------------------------------------------------

Results: 17 row(s)

| name                                | geekRating |
| ----------------------------------- | ---------- |
| Gloomhaven                          | 8.335      |
| War of the Ring: Second Edition     | 8.192      |
| Spirit Island                       | 8.144      |
| Arkham Horror: The Card Game        | 7.899      |
| Terra Mys

"Here are the 17 Fantasy-category games with a geek rating above 7.5, ranked from highest to lowest:\n\n1. **Gloomhaven** — 8.34\n2. **War of the Ring: Second Edition** — 8.19\n3. **Spirit Island** — 8.14\n4. **Arkham Horror: The Card Game** — 7.90\n5. **Terra Mystica** — 7.88\n6. **Too Many Bones** — 7.87\n7. **Mage Knight Board Game** — 7.85\n8. **Caverna: The Cave Farmers** — 7.75\n9. **Blood Rage** — 7.74\n10. **Mansions of Madness: Second Edition** — 7.70\n11. **Mechs vs. Minions** — 7.65\n12. **Kingdom Death: Monster** — 7.64\n13. **Five Tribes: The Djinns of Naqala** — 7.62\n14. **Clank!: A Deck-Building Adventure** — 7.61\n15. **Aeon's End** — 7.60\n16. **Lords of Waterdeep** — 7.59\n17. **Eldritch Horror** — 7.50\n\nGloomhaven leads the list, with several heavy hitters like War of the Ring and Spirit Island also scoring above 8."

In [17]:
ask("Which fake players gave a rating of 10 to any game, and what games did they rate?")

Question: Which fake players gave a rating of 10 to any game, and what games did they rate?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX fake: <https://vejdemo.se/boardgames/fake#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?playerName ?gameName ?personalRating WHERE {
  ?op a bgg:PlayerOpinion ;
      bgg:hasOpinionHolder ?player ;
      bgg:hasOpinionOf ?game ;
      bgg:hasPlayerRatingOpinion ?personalRating .
  ?player rdfs:label ?playerName .
  ?game bgg:hasName ?gameName .
  FILTER(STRSTARTS(STR(?player), STR(fake:)))
  FILTER(?personalRating = 10)
}
ORDER BY ?playerName ?gameName
------------------------------------------------------------

Results: 418 row(s) (truncated to 50 for interpretation)

| playerName         | gameName                                      | personalRating |
| ------------------ | --------------------------------------------- | 

"There are **418 total ratings of 10** given by fake players across various games (showing only the top 50 results).\n\nHere's a sample of the fake players who rated games a perfect 10:\n\n- **Akiko Petit**: Papa Bear, Plucky Pilots\n- **Akiko Walker**: Fall of Magic\n- **Alejandro Martínez**: Seals, The Red Dragon Inn: Allies – Spyke and Flower\n- **Alejandro Scott**: Lie Detector, Ringo, Skyscrapers\n- **Anders Patel**: Mario Party-e, Planet Apocalypse, Sweetlandia\n- **Andrei Torres**: Raccoon Tycoon, Run Wild\n- **Arjun Schneider**: Ramses Return, Sucking Vacuum\n- **Barbara Adams**: Lost Woods, Quest for the Auburn Pelt, Tara Wolf in Valley of the Kings\n- **Bjorn Baker**: 1829, 18Lilliput, La Petite Mort, Operation, Tricky Druids\n- **Britta Fischer**: Ciao Ciao..., Holyland, IUNU, Termity\n\nMany fake players gave multiple perfect ratings, with Bjorn Baker and Britta Fischer notably each rating 4+ games as 10/10. Since results are truncated, there are 368 additional rating entri

## Your Question

In [12]:
# Change this and run
ask("What are the best cooperative games?")


Question: What are the best cooperative games?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  { ?game bgg:hasCategory ?v } UNION { ?game bgg:hasMechanic ?v }
  ?v rdfs:label|skos:altLabel ?label .
  FILTER(CONTAINS(LCASE(str(?label)), LCASE("co-operative")) || CONTAINS(LCASE(str(?label)), LCASE("cooperative")))
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------

Results: 10 row(s)

| name                                                    | rating |
| ------------------------------------------------------- | ------ |
| TerroriXico                                             | 9.43   |
| War Titans: Invaders Must Die!     

'Here are the top 10 cooperative board games by rating:\n\n1. **TerroriXico** — 9.43\n2. **War Titans: Invaders Must Die!** — 9.23\n3. **Oathsworn: Into the Deepwood** — 8.90\n4. **Mage Knight: Ultimate Edition** — 8.86\n5. **Star Trek: Alliance – Dominion War Campaign** — 8.70\n6. **Dungeon Universalis** — 8.69\n7. **Too Many Bones: Undertow** — 8.68\n8. **League of Dungeoneers** — 8.66\n9. **Rangers of Shadow Deep: Standard Edition** — 8.65\n10. **Shadows of Brimstone: Swamps of Death (Revised Edition)** — 8.59\n\nThese games span a range of styles, from epic fantasy adventures (Oathsworn, Mage Knight) to dungeon crawlers (Dungeon Universalis, League of Dungeoneers) and themed campaigns (Star Trek: Alliance). Oathsworn and Mage Knight are particularly well-known standouts in the cooperative gaming community.'

In [ ]:
# Change this and run
ask("I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?")
### This is a great test to see if the agent will hallucinate when it doesn't have data. It does not!

Question: I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?name ?rating (COUNT(DISTINCT ?sharedTag) AS ?overlap) WHERE {
  {
    ?seed a bgg:Game ;
          bgg:hasName "Set"^^xsd:string .
  } UNION {
    ?seed a bgg:Game ;
          bgg:hasName "Prosperitea"^^xsd:string .
  }
  { ?seed bgg:hasCategory ?sharedTag } UNION { ?seed bgg:hasMechanic ?sharedTag }
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  { ?game bgg:hasCategory ?sharedTag } UNION { ?game bgg:hasMechanic ?sharedTag }
  FILTER(?name != "Set"^^xsd:string && ?name != "Prosperitea"^^xsd:string)
}
GROUP BY ?name ?rating
ORDER BY DESC(?overlap) DESC(?rating)
LIMIT 2

'I couldn\'t find any games with similar mechanics or categories to *Set* and *Prosperitea* in the database.\n\nThis could be because:\n- One or both of these games may not be present in the dataset, or their names might be stored differently (e.g., with different capitalization, punctuation, or edition suffixes).\n- The games may not have category or mechanic tags assigned in the data.\n\nYou might try searching for variations of the titles (for example, "SET" or "Prosperi-Tea Planning") to confirm they exist in the database, then re-run the query.'

In [ ]:
ask("I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?")
### this breaks! Fun! figure out why and fix it! (hint: check the mechanics and categories of these games in the data, and see how the query is structured to find similar games)

Question: I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT DISTINCT ?name ?rating WHERE {
  {
    ?seed a bgg:Game ;
          bgg:hasName "Set"^^xsd:string .
  } UNION {
    ?seed a bgg:Game ;
          bgg:hasName "Ricochet Robots"^^xsd:string .
  }
  { ?seed bgg:hasMechanic ?tag . ?game bgg:hasMechanic ?tag . }
  UNION
  { ?seed bgg:hasCategory ?tag . ?game bgg:hasCategory ?tag . }
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating .
  FILTER(?game != ?seed)
  FILTER(?name != "Set"^^xsd:string && ?name != "Ricochet Robots"^^xsd:string)
}
ORDER BY DESC(?rating)
LIMIT 20
------------------------------------------------------------



In [22]:
ask("I want an 8 player game that is not a party game. What are the best options?")

Question: I want an 8 player game that is not a party game. What are the best options?

Generated SPARQL:
------------------------------------------------------------
PREFIX bgg: <https://raw.githubusercontent.com/susvej/bg_ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT DISTINCT ?name ?rating WHERE {
  ?game a bgg:Game ;
        bgg:hasName ?name ;
        bgg:hasRating ?rating ;
        bgg:hasMinPlayers ?min ;
        bgg:hasMaxPlayers ?max .
  FILTER(?min <= 8 && ?max >= 8)
  FILTER NOT EXISTS {
    ?game bgg:hasCategory ?cat .
    ?cat rdfs:label ?catLabel .
    FILTER(CONTAINS(LCASE(str(?catLabel)), LCASE("party game")))
  }
}
ORDER BY DESC(?rating) LIMIT 10
------------------------------------------------------------



ReadTimeout: HTTPConnectionPool(host='localhost', port=7200): Read timed out. (read timeout=30)